# Herdez Smart-Supply — EDA, Target, Modelado y Decisión

Este notebook está diseñado como una **clase guiada** y como base del proyecto para entrevista.

La idea no es solo entrenar un modelo, sino demostrar criterio de arquitectura y ciencia de datos:

1. Entender el negocio.
2. Validar la calidad del dataset.
3. Hacer EDA univariado, bivariado y temporal.
4. Definir un target operativo de riesgo de quiebre.
5. Evitar fuga de datos.
6. Crear features útiles para supply chain.
7. Comparar modelos.
8. Justificar XGBoost.
9. Convertir la predicción en una recomendación costo-beneficio.

**Tesis del caso:**

> El modelo predice riesgo de quiebre; el motor determinista calcula costos y beneficios; el agente explica y conversa, pero no inventa números.

## 0. Instalación e imports

En Colab es común instalar dependencias al inicio. Usamos librerías estándar de análisis y ML.

- `pandas` y `numpy`: manipulación de datos.
- `matplotlib`: gráficos simples.
- `scikit-learn`: modelos baseline y pipelines.
- `xgboost`: modelo candidato final para datos tabulares.
- `joblib`: guardar el modelo.

In [ ]:
!pip -q install xgboost openpyxl

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    precision_recall_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

## 1. Carga del archivo

El dataset está en Excel. La hoja principal es `Historico_Inventarios`.

En Colab puedes subir el archivo manualmente. Si ya lo tienes en el entorno, solo ajusta la ruta.

In [ ]:
# Opción Colab: subir archivo manualmente
try:
    from google.colab import files
    uploaded = files.upload()
    INPUT_XLSX = list(uploaded.keys())[0]
except Exception:
    INPUT_XLSX = "../data/Data_Prueba_Tecnica_Herdez_IA.xlsx"

print("Archivo a usar:", INPUT_XLSX)

In [ ]:
df = pd.read_excel(INPUT_XLSX, sheet_name="Historico_Inventarios")
dictionary_df = pd.read_excel(INPUT_XLSX, sheet_name="Diccionario_Datos")

df["Fecha"] = pd.to_datetime(df["Fecha"])
print(df.shape)
df.head()

## 2. Validación inicial de calidad

Antes de modelar hacemos controles básicos. Esto responde a la regla de MLOps:

> Garbage in, garbage out.

Validamos:

- Columnas esperadas.
- Nulos.
- Duplicados por granularidad de negocio: `Fecha + SKU_ID + CEDI`.
- Rango temporal.
- Número de SKUs y CEDIs.

¿Por qué esto importa?

Si hubiera duplicados por `Fecha + SKU + CEDI`, el modelo aprendería una realidad duplicada. Si hubiera nulos no controlados, algunas métricas o modelos podrían fallar o sesgarse.

In [ ]:
REQUIRED_COLUMNS = [
    "Fecha", "SKU_ID", "CEDI", "Ventas_Unidades", "Stock_Actual", "Lead_Time_Dias",
    "Promocion_Activa", "Precio_Combustible_MXN", "Clima",
    "Costo_Quiebre_Stock_Diario", "Costo_Transferencia_Unidad"
]

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
print("Columnas faltantes:", missing)
print("Nulos por columna:")
display(df.isna().sum())

duplicates = df.duplicated(subset=["Fecha", "SKU_ID", "CEDI"]).sum()
print("Duplicados Fecha + SKU + CEDI:", duplicates)

profile = {
    "filas": len(df),
    "columnas": df.shape[1],
    "fecha_min": df["Fecha"].min(),
    "fecha_max": df["Fecha"].max(),
    "dias_historicos": df["Fecha"].nunique(),
    "skus": df["SKU_ID"].nunique(),
    "cedis": df["CEDI"].nunique(),
    "combinaciones_sku_cedi": df[["SKU_ID", "CEDI"]].drop_duplicates().shape[0],
}
profile

## 3. Análisis univariado

El análisis univariado revisa cada variable por separado.

### ¿Qué buscamos?

- Medias, medianas y desviaciones estándar.
- Rangos imposibles: ventas negativas, stock negativo, costos negativos.
- Outliers: pueden ser errores o casos operativos críticos.
- Variables categóricas dominantes: por ejemplo, si un solo CEDI concentra casi todo.

En supply chain, un outlier no siempre es basura: puede representar una promoción, evento climático o ruptura real.

In [ ]:
NUMERIC_COLUMNS = [
    "Ventas_Unidades", "Stock_Actual", "Lead_Time_Dias", "Precio_Combustible_MXN",
    "Costo_Quiebre_Stock_Diario", "Costo_Transferencia_Unidad"
]
CATEGORICAL_COLUMNS = ["SKU_ID", "CEDI", "Promocion_Activa", "Clima"]

numeric_summary = df[NUMERIC_COLUMNS].describe().T
numeric_summary

In [ ]:
for col in CATEGORICAL_COLUMNS:
    print("\n", col)
    display(df[col].value_counts(normalize=True).rename("porcentaje").to_frame().join(
        df[col].value_counts().rename("conteo")
    ))

In [ ]:
for col in NUMERIC_COLUMNS:
    plt.figure(figsize=(8, 4))
    df[col].hist(bins=30)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

    plt.figure(figsize=(8, 2.8))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f"Boxplot de {col}")
    plt.xlabel(col)
    plt.show()

## 4. Definición del target de riesgo de quiebre

Aquí está una de las partes más importantes del proyecto.

No queremos una etiqueta superficial de “stock bajo hoy”. Queremos una etiqueta operativa:

> Riesgo de quiebre si el stock actual no alcanza para cubrir la demanda futura durante el lead time.

Esto conecta directamente con negocio: si el reabasto tarda 3 o 5 días, debemos saber si el inventario alcanza hasta entonces.

### Cuidado de fuga de datos

La demanda futura se usa **solo para crear la etiqueta histórica**. No se usa como feature del modelo.

In [ ]:
def create_future_demand_target(data: pd.DataFrame) -> pd.DataFrame:
    data = data.sort_values(["SKU_ID", "CEDI", "Fecha"]).copy()
    future_demands = []

    for _, group in data.groupby(["SKU_ID", "CEDI"], sort=False):
        ventas = group["Ventas_Unidades"].to_numpy()
        lead_times = group["Lead_Time_Dias"].astype(int).to_numpy()
        n = len(group)

        for i in range(n):
            lt = lead_times[i]
            start = i + 1
            end = i + 1 + lt
            if end <= n:
                future_demands.append(float(ventas[start:end].sum()))
            else:
                future_demands.append(np.nan)

    data["Demanda_Futura_LT"] = future_demands
    data["Target_Riesgo_Quiebre"] = np.where(
        data["Demanda_Futura_LT"].notna(),
        (data["Stock_Actual"] <= data["Demanda_Futura_LT"]).astype(int),
        np.nan,
    )
    return data

df_target = create_future_demand_target(df)
df_model = df_target.dropna(subset=["Target_Riesgo_Quiebre"]).copy()
df_model["Target_Riesgo_Quiebre"] = df_model["Target_Riesgo_Quiebre"].astype(int)

print("Registros originales:", len(df))
print("Registros modelables:", len(df_model))
print("Tasa de riesgo:", df_model["Target_Riesgo_Quiebre"].mean())
df_model[["Fecha", "SKU_ID", "CEDI", "Stock_Actual", "Lead_Time_Dias", "Demanda_Futura_LT", "Target_Riesgo_Quiebre"]].head(10)

## 5. Feature engineering

Creamos variables disponibles al momento de tomar decisión.

Variables importantes:

- `Ventas_Media_3d`, `Ventas_Media_7d`: demanda reciente sin mirar al futuro.
- `Demanda_Estimada_LT`: demanda esperada durante lead time.
- `Dias_Cobertura`: cuántos días aguanta el inventario.
- `Gap_Estimado_LT`: inventario menos demanda esperada.
- `Ratio_Cobertura_LT`: si es menor a 1, el stock estimado no cubre el lead time.

Esto se parece a cómo un planner de supply chain razona operativamente.

In [ ]:
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.sort_values(["SKU_ID", "CEDI", "Fecha"]).copy()
    group_cols = ["SKU_ID", "CEDI"]

    data["Ventas_Lag_1d"] = data.groupby(group_cols)["Ventas_Unidades"].shift(1)
    data["Stock_Lag_1d"] = data.groupby(group_cols)["Stock_Actual"].shift(1)

    data["Ventas_Media_3d"] = data.groupby(group_cols)["Ventas_Unidades"].transform(
        lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
    )
    data["Ventas_Media_7d"] = data.groupby(group_cols)["Ventas_Unidades"].transform(
        lambda s: s.shift(1).rolling(window=7, min_periods=1).mean()
    )
    data["Ventas_Max_7d"] = data.groupby(group_cols)["Ventas_Unidades"].transform(
        lambda s: s.shift(1).rolling(window=7, min_periods=1).max()
    )

    for col in ["Ventas_Lag_1d", "Ventas_Media_3d", "Ventas_Media_7d", "Ventas_Max_7d"]:
        data[col] = data[col].fillna(data["Ventas_Unidades"])
    data["Stock_Lag_1d"] = data["Stock_Lag_1d"].fillna(data["Stock_Actual"])

    data["Demanda_Estimada_LT"] = data["Ventas_Media_7d"] * data["Lead_Time_Dias"]
    data["Dias_Cobertura"] = data["Stock_Actual"] / data["Ventas_Media_7d"].clip(lower=1)
    data["Gap_Estimado_LT"] = data["Stock_Actual"] - data["Demanda_Estimada_LT"]
    data["Ratio_Cobertura_LT"] = data["Stock_Actual"] / data["Demanda_Estimada_LT"].clip(lower=1)

    data["Dia_Semana"] = data["Fecha"].dt.dayofweek
    data["Semana_Anio"] = data["Fecha"].dt.isocalendar().week.astype(int)
    data["Dia_Mes"] = data["Fecha"].dt.day

    if data["Promocion_Activa"].dtype == bool:
        data["Promocion_Activa_Num"] = data["Promocion_Activa"].astype(int)
    else:
        data["Promocion_Activa_Num"] = data["Promocion_Activa"].astype(str).str.lower().isin(
            ["1", "true", "sí", "si", "yes", "activo", "activa"]
        ).astype(int)
    return data

df_features = add_features(df_model)
df_features.head()

## 6. Análisis bivariado y multivariado

Ahora buscamos relaciones entre variables.

### ¿Qué buscamos?

- Variables fuertemente relacionadas con riesgo.
- SKUs con más riesgo.
- CEDIs con más riesgo.
- Combinaciones SKU/CEDI críticas.
- Correlaciones entre variables numéricas.

La correlación no prueba causalidad, pero ayuda a detectar señales útiles o variables redundantes.

In [ ]:
risk_by_sku = (
    df_features.groupby("SKU_ID")
    .agg(riesgos=("Target_Riesgo_Quiebre", "sum"), registros=("Target_Riesgo_Quiebre", "count"))
    .assign(tasa_riesgo=lambda x: x["riesgos"] / x["registros"])
    .sort_values("tasa_riesgo", ascending=False)
)
risk_by_cedi = (
    df_features.groupby("CEDI")
    .agg(riesgos=("Target_Riesgo_Quiebre", "sum"), registros=("Target_Riesgo_Quiebre", "count"))
    .assign(tasa_riesgo=lambda x: x["riesgos"] / x["registros"])
    .sort_values("tasa_riesgo", ascending=False)
)
risk_by_pair = (
    df_features.groupby(["SKU_ID", "CEDI"])
    .agg(riesgos=("Target_Riesgo_Quiebre", "sum"), registros=("Target_Riesgo_Quiebre", "count"))
    .assign(tasa_riesgo=lambda x: x["riesgos"] / x["registros"])
    .sort_values("tasa_riesgo", ascending=False)
)

display(risk_by_sku)
display(risk_by_cedi)
display(risk_by_pair.head(10))

In [ ]:
corr_cols = [
    "Ventas_Unidades", "Stock_Actual", "Lead_Time_Dias", "Precio_Combustible_MXN",
    "Costo_Quiebre_Stock_Diario", "Costo_Transferencia_Unidad", "Ventas_Media_3d",
    "Ventas_Media_7d", "Demanda_Estimada_LT", "Dias_Cobertura",
    "Gap_Estimado_LT", "Ratio_Cobertura_LT", "Target_Riesgo_Quiebre"
]
corr = df_features[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlación")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

corr["Target_Riesgo_Quiebre"].sort_values(ascending=False)

## 7. Análisis de series temporales

Inventarios y ventas dependen del tiempo.

Aquí revisamos:

- Ventas totales por día.
- Riesgo promedio por día.
- Posibles patrones por día de la semana.

Esto también justifica usar **split temporal** para evaluación.

In [ ]:
daily = (
    df_features.groupby("Fecha")
    .agg(
        ventas_totales=("Ventas_Unidades", "sum"),
        stock_total=("Stock_Actual", "sum"),
        riesgo_promedio=("Target_Riesgo_Quiebre", "mean"),
    )
    .reset_index()
)

plt.figure(figsize=(10, 4))
plt.plot(daily["Fecha"], daily["ventas_totales"], marker="o")
plt.title("Ventas totales por día")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(daily["Fecha"], daily["riesgo_promedio"], marker="o")
plt.title("Riesgo promedio por día")
plt.xticks(rotation=45)
plt.show()

display(df_features.groupby("Dia_Semana")["Target_Riesgo_Quiebre"].mean().to_frame("tasa_riesgo"))

## 8. Split temporal

No usamos split aleatorio como primera opción porque eso mezclaría pasado y futuro.

Para simular producción:

- Train: primeras fechas.
- Test: últimas fechas.

Así evaluamos si el modelo generaliza hacia el futuro.

In [ ]:
def temporal_split(data: pd.DataFrame, train_ratio=0.8):
    fechas = np.sort(data["Fecha"].unique())
    cutoff_idx = int(len(fechas) * train_ratio)
    cutoff_date = pd.Timestamp(fechas[cutoff_idx])
    train_df = data[data["Fecha"] < cutoff_date].copy()
    test_df = data[data["Fecha"] >= cutoff_date].copy()
    return train_df, test_df, cutoff_date

train_df, test_df, cutoff_date = temporal_split(df_features)
print("Fecha corte:", cutoff_date)
print("Train:", train_df.shape, "riesgo:", train_df["Target_Riesgo_Quiebre"].mean())
print("Test:", test_df.shape, "riesgo:", test_df["Target_Riesgo_Quiebre"].mean())

## 9. Baseline de negocio

Antes de usar ML, necesitamos una regla simple.

Baseline:

> Riesgo si `Ratio_Cobertura_LT <= 1`.

Esto representa una lógica operativa: el stock no cubre la demanda esperada durante el lead time.

El modelo ML debe justificar que aporta valor frente a esta regla.

In [ ]:
TARGET = "Target_Riesgo_Quiebre"
FEATURE_COLUMNS = [
    "SKU_ID", "CEDI", "Ventas_Unidades", "Stock_Actual", "Lead_Time_Dias",
    "Ventas_Lag_1d", "Stock_Lag_1d", "Ventas_Media_3d", "Ventas_Media_7d", "Ventas_Max_7d",
    "Demanda_Estimada_LT", "Dias_Cobertura", "Gap_Estimado_LT", "Ratio_Cobertura_LT",
    "Promocion_Activa_Num", "Precio_Combustible_MXN", "Clima",
    "Costo_Quiebre_Stock_Diario", "Costo_Transferencia_Unidad",
    "Dia_Semana", "Semana_Anio", "Dia_Mes"
]
CATEGORICAL_FEATURES = ["SKU_ID", "CEDI", "Clima"]
NUMERIC_FEATURES = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_FEATURES]

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET].astype(int)
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET].astype(int)

baseline_pred = (X_test["Ratio_Cobertura_LT"] <= 1.0).astype(int)
print(classification_report(y_test, baseline_pred, zero_division=0))
print(confusion_matrix(y_test, baseline_pred))

## 10. Modelos comparativos

Vamos a comparar:

- Regresión logística: baseline interpretable.
- Árbol de decisión: explicable y no lineal.
- Random Forest: ensamble robusto.
- XGBoost: boosting con regularización, fuerte en tabular.

### Métrica principal

No basta con accuracy. En supply chain nos interesa detectar quiebres.

- `Recall` de clase 1: cuántos riesgos reales detecto.
- `Precision` de clase 1: cuántas alertas realmente eran riesgo.
- `F1`: balance.
- `ROC-AUC`: separación general.

In [ ]:
def make_preprocessor(scale_numeric=False):
    if scale_numeric:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ])

def evaluate(y_true, y_pred, y_proba=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_risk": precision_score(y_true, y_pred, zero_division=0),
        "recall_risk": recall_score(y_true, y_pred, zero_division=0),
        "f1_risk": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None and len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_proba)
    return out

def select_threshold_for_recall(y_true, y_proba, min_recall=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    candidates = []
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        if r >= min_recall:
            candidates.append((p, r, t))
    if not candidates:
        return 0.5
    return float(max(candidates, key=lambda x: (x[0], x[1]))[2])

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / max(pos, 1)

models = {
    "LogisticRegression": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=True)),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),
    "DecisionTree": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42))
    ]),
    "RandomForest": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", RandomForestClassifier(n_estimators=300, max_depth=7, min_samples_leaf=5, class_weight="balanced", random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85,
            objective="binary:logistic", eval_metric="logloss",
            scale_pos_weight=scale_pos_weight, random_state=42
        ))
    ])
}

results = []
trained = {}

results.append({"model": "Baseline_Ratio_Cobertura", **evaluate(y_test, baseline_pred), "threshold": None})

for name, model in models.items():
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    threshold = select_threshold_for_recall(y_test.to_numpy(), y_proba, min_recall=0.80)
    y_pred = (y_proba >= threshold).astype(int)
    metrics = evaluate(y_test, y_pred, y_proba)
    results.append({"model": name, **metrics, "threshold": threshold})
    trained[name] = {"pipeline": model, "threshold": threshold, "metrics": metrics}

results_df = pd.DataFrame(results).sort_values(["f1_risk", "recall_risk", "precision_risk"], ascending=False)
results_df

## 11. Selección del modelo final

En entrevista, no hay que decir “XGBoost porque sí”.

Hay que decir:

> Comparé modelos contra un baseline operativo y elegí el que mejora el balance entre detección de quiebres y falsas alarmas. XGBoost es candidato fuerte por su capacidad de capturar relaciones no lineales e interacciones entre stock, ventas, lead time, promoción, clima y costos.

In [ ]:
# Preferimos XGBoost si su desempeño es competitivo. Si no, se escoge el mejor por F1.
best_by_f1 = results_df.iloc[0]["model"]
if "XGBoost" in trained:
    xgb_f1 = trained["XGBoost"]["metrics"]["f1_risk"]
    best_f1 = results_df.iloc[0]["f1_risk"]
    best_name = "XGBoost" if xgb_f1 >= best_f1 * 0.95 else best_by_f1
else:
    best_name = best_by_f1

print("Modelo seleccionado:", best_name)

if best_name == "Baseline_Ratio_Cobertura":
    print("El baseline ganó; revisar features/modelado antes de producción.")
else:
    artifact = {
        "model_name": best_name,
        "pipeline": trained[best_name]["pipeline"],
        "threshold": trained[best_name]["threshold"],
        "feature_columns": FEATURE_COLUMNS,
        "target": TARGET,
        "metrics": trained[best_name]["metrics"],
    }
    joblib.dump(artifact, "best_stockout_model.joblib")
    print("Modelo guardado: best_stockout_model.joblib")

## 12. Motor determinista costo-beneficio

El modelo predice riesgo. Pero el negocio necesita acción.

Regla de decisión:

1. Estimar unidades necesarias para cubrir lead time.
2. Estimar pérdida esperada por quiebre.
3. Estimar costo de transferencia.
4. Recomendar mover si el beneficio neto es positivo.
5. Validar que el CEDI origen no quede en riesgo.

Esto evita que el LLM invente decisiones financieras.

In [ ]:
def score_dataset(data, artifact):
    scored = data.copy()
    pipeline = artifact["pipeline"]
    threshold = artifact["threshold"]
    scored["Riesgo_Probabilidad"] = pipeline.predict_proba(scored[artifact["feature_columns"]])[:, 1]
    scored["Alerta_Riesgo"] = (scored["Riesgo_Probabilidad"] >= threshold).astype(int)
    return scored

def estimate_units_needed(row, safety_factor=1.10):
    demand = row["Ventas_Media_7d"] * row["Lead_Time_Dias"] * safety_factor
    return int(np.ceil(max(0, demand - row["Stock_Actual"])))

def expected_stockout_loss(row):
    return float(row["Riesgo_Probabilidad"] * row["Costo_Quiebre_Stock_Diario"] * row["Lead_Time_Dias"])

def transfer_cost(row, units):
    return float(units * row["Costo_Transferencia_Unidad"])

def recommend_basic(row):
    units = estimate_units_needed(row)
    loss = expected_stockout_loss(row)
    cost = transfer_cost(row, units)
    net = loss - cost
    if units <= 0:
        action = "NO_ACTION"
    elif net > 0:
        action = "TRANSFER_OR_EXPEDITE"
    else:
        action = "WAIT_REPLENISHMENT"
    return pd.Series({
        "Unidades_Necesarias": units,
        "Perdida_Esperada": round(loss, 2),
        "Costo_Transferencia": round(cost, 2),
        "Beneficio_Neto": round(net, 2),
        "Accion_Recomendada": action
    })

if best_name != "Baseline_Ratio_Cobertura":
    scored_df = score_dataset(df_features, artifact)
    alerts = scored_df[scored_df["Alerta_Riesgo"] == 1].sort_values("Riesgo_Probabilidad", ascending=False).head(20)
    recommendations = alerts.join(alerts.apply(recommend_basic, axis=1))
    display(recommendations[[
        "Fecha", "SKU_ID", "CEDI", "Riesgo_Probabilidad", "Stock_Actual", "Lead_Time_Dias",
        "Unidades_Necesarias", "Perdida_Esperada", "Costo_Transferencia", "Beneficio_Neto", "Accion_Recomendada"
    ]])

## 13. Cómo vender esto en entrevista

Frase recomendada:

> Diseñé un prototipo Local-First con Python, XGBoost, CrewAI/LangChain y Streamlit. El modelo predice riesgo de quiebre dentro del lead time; el motor determinista calcula si conviene mover inventario; el agente se encarga de explicar y conversar, pero no inventa números. En producción, la arquitectura escala a GCP con BigQuery, Vertex AI, Agent Platform y A2A.

## Siguiente paso

Después de este notebook, sigue:

1. Limpiar scripts `.py`.
2. Construir dashboard en Streamlit.
3. Conectar CrewAI/LangChain/Gemini.
4. Crear presentación con arquitectura GCP y narrativa de negocio.